In [2]:
#!/mrhome/amingk/anaconda3/envs/7tpd/bin/python
import numpy as np
import pandas as pd
import stan
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('/mrhome/amingk/Documents/7TPD/ActStimRL')
import arviz as az
from scipy.stats import gaussian_kde
from utils import *
import os
from scipy.stats import pearsonr
from scipy import stats
import statsmodels.api as sm

In [3]:
# read behavioral data with summary paramter
df_beh_paramerer = pd.read_csv(PROJECT_NoNAN_BEH_REL_IRREL_HIGH_REWARD_OPTION_GROUPBY_ALL_FILE_MODEL_PARAMETER)
# drop group column 
df_beh_paramerer.drop('group', axis=1, inplace=True)
# Read clinical data
clinical_evaluation = pd.read_csv(f'{PROJECT_CLIN_EVAL_FILE}')

In [4]:
# Split clinical data
clinical_evaluation_PD = clinical_evaluation[clinical_evaluation['patient'] == 'PD']
clinical_evaluation_HC = clinical_evaluation[clinical_evaluation['patient'] == 'HC']
# PD wih OFF and ON
clinical_evaluation_PD_OFF = clinical_evaluation_PD.drop(['most_affected_left_right_side_UPDRSON', 'total_UPDRSON'], axis=1)
clinical_evaluation_PD_ON = clinical_evaluation_PD.drop(['most_affected_left_right_side_UPDRSOFF', 'total_UPDRSOFF'], axis=1)

# rename of columns in HC
clinical_evaluation_HC = clinical_evaluation_HC.drop(['most_affected_left_right_side_UPDRSOFF', 'total_UPDRSOFF'], axis=1)
clinical_evaluation_HC.rename(columns={'most_affected_left_right_side_UPDRSON':'most_affected_left_right_side_UPDRS',
                                       'total_UPDRSON':'total_UPDRS'}, inplace=True)
# rename of columns in PD
clinical_evaluation_PD_ON.rename(columns={'most_affected_left_right_side_UPDRSON':'most_affected_left_right_side_UPDRS',
                                          'total_UPDRSON':'total_UPDRS'}, inplace=True)
clinical_evaluation_PD_OFF.rename(columns={'most_affected_left_right_side_UPDRSOFF':'most_affected_left_right_side_UPDRS',
                                           'total_UPDRSOFF':'total_UPDRS'}, inplace=True)


# Filter behavioral data
df_PD_Act_OFF = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'PD') &
    (df_beh_paramerer['block'] == 'Act') &
    (df_beh_paramerer['medication'] == 'OFF')]

df_PD_Stim_OFF = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'PD') &
    (df_beh_paramerer['block'] == 'Stim') &
    (df_beh_paramerer['medication'] == 'OFF')]

df_PD_Act_ON = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'PD') &
    (df_beh_paramerer['block'] == 'Act') &
    (df_beh_paramerer['medication'] == 'ON')]

df_PD_Stim_ON = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'PD') &
    (df_beh_paramerer['block'] == 'Stim') &
    (df_beh_paramerer['medication'] == 'ON')]

df_HC_Act = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'HC') &
    (df_beh_paramerer['block'] == 'Act')]

df_HC_Stim = df_beh_paramerer[
    (df_beh_paramerer['patient'] == 'HC') &
    (df_beh_paramerer['block'] == 'Stim')]

# Merge behavioral + clinical
merged_HC_Act = pd.merge(df_HC_Act, clinical_evaluation_HC, on=['sub_ID','patient'], how='inner')
merged_HC_Stim = pd.merge(df_HC_Stim, clinical_evaluation_HC, on=['sub_ID','patient'], how='inner')

merged_PD_Act_OFF = pd.merge(df_PD_Act_OFF, clinical_evaluation_PD_OFF, on=['sub_ID','patient'], how='inner')
merged_PD_Stim_OFF = pd.merge(df_PD_Stim_OFF, clinical_evaluation_PD_OFF, on=['sub_ID','patient'], how='inner')

merged_PD_Act_ON = pd.merge(df_PD_Act_ON, clinical_evaluation_PD_ON, on=['sub_ID','patient'], how='inner')
merged_PD_Stim_ON = pd.merge(df_PD_Stim_ON, clinical_evaluation_PD_ON, on=['sub_ID','patient'], how='inner')

# Concatenate all
df_all = pd.concat([
    merged_HC_Act,
    merged_HC_Stim,
    merged_PD_Act_OFF,
    merged_PD_Stim_OFF,
    merged_PD_Act_ON,
    merged_PD_Stim_ON
], ignore_index=True)

# rename some columns
df_all = df_all.rename(columns={'block':'condition'})
df_all['condition']=df_all['condition'].map({'Stim':'Clr', 'Act':'Act'})
# save in csv 
df_all.to_csv(f'{PROJECT_CLIN_EVAL_PARAM_FILE}', index=False )

In [5]:
## make a groupby and average across medication and condition
#df_HC_PD_Avr_Meds_Conds = df_all.groupby(['patient', 'sub_ID'])[['age', 'relevantHighRewardOption', 'irrelevantHighRewardOption',
#                                                  'relevantVsIrrelevantHighRewardOption', 'weight_parameter_map',
#                                                  'weight_parameter_map_magnitude', 'MoCA', 'BDI', 'LARS']].mean().reset_index()
#
## save in csv 
#df_HC_PD_Avr_Meds_Conds.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_HC_PD_Avr_Meds_Conds.csv', index=False )
#
#
## make a groupby and average across medication
#df_HC_PD_Avr_Meds = df_all.groupby(['patient', 'sub_ID', 'block'])[['age', 'relevantHighRewardOption', 'irrelevantHighRewardOption',
#                                                                    'relevantVsIrrelevantHighRewardOption', 'weight_parameter_map',
#                                                                    'weight_parameter_map_magnitude', 'MoCA', 'BDI', 'LARS']].mean().reset_index()
#
## save in csv 
#df_HC_PD_Avr_Meds.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_HC_PD_Avr_Meds.csv', index=False )
#
## make a groupby for Act
#df_HC_PD_Act_Avr_Meds = df_all[df_all['block']=='Act'].groupby(['patient', 'sub_ID', 'block'])[['age', 'relevantHighRewardOption', 'irrelevantHighRewardOption',
#                                                                          'relevantVsIrrelevantHighRewardOption', 'weight_parameter_map',
#                                                                          'weight_parameter_map_magnitude', 'MoCA', 'BDI', 'LARS']].mean().reset_index()
#
## save in csv 
#df_HC_PD_Act_Avr_Meds.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_HC_PD_Act_Avr_Meds.csv', index=False )
#
## make a groupby for Clr
#df_HC_PD_Clr_Avr_Meds = df_all[df_all['block']=='Clr'].groupby(['patient', 'sub_ID', 'block'])[['age', 'relevantHighRewardOption', 'irrelevantHighRewardOption',
#                                                                               'relevantVsIrrelevantHighRewardOption', 'weight_parameter_map',
#                                                                               'weight_parameter_map_magnitude', 'MoCA', 'BDI', 'LARS']].mean().reset_index()
## save in csv 
#df_HC_PD_Clr_Avr_Meds.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_HC_PD_Clr_Avr_Meds.csv', index=False )
#
#
## make a groupby for PD average across medication and condition
#df_PD_Avr_Meds_Conds = df_all[df_all['patient']=='PD'].groupby(['patient', 'sub_ID', 'x_number', 'sex'])[['relevantHighRewardOption', 'irrelevantHighRewardOption',
#                                                                                       'relevantVsIrrelevantHighRewardOption', 'weight_parameter_map',
#                                                                                       'weight_parameter_map_magnitude', 'age',
#                                                                                       'disease_duration', 'time_symptomns',
#                                                                                       'systolic_blood_pressure_baselineUPDRS', 'total_BaselineUPDRS',
#                                                                                       'total_UPDRS', 'NMSS', 'MoCA', 'BDI', 'LARS']].mean().reset_index()
## save in csv 
#df_PD_Avr_Meds_Conds.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_PD_Avr_Meds_Conds.csv', index=False )


In [6]:
# PD OFF
df_PD_OFF_Act = df_all[(df_all['patient'] == 'PD') &
                       (df_all['medication'] == 'OFF') &
                       (df_all['condition'] == 'Act')].copy().reset_index(drop=True)

# PD ON
df_PD_ON_Act = df_all[(df_all['patient'] == 'PD') &
                      (df_all['medication'] == 'ON') &
                      (df_all['condition'] == 'Act')].copy().reset_index(drop=True)

# ensure same subject order
df_PD_OFF_Act = df_PD_OFF_Act.sort_values('sub_ID').reset_index(drop=True)
df_PD_ON_Act = df_PD_ON_Act.sort_values('sub_ID').reset_index(drop=True)

# New dataframe
df_PD_OFFvsON_Act = df_PD_OFF_Act[['patient', 'medication', 'sub_ID', 'condition', 'x_number']].copy()

# Compute contrasts (ON - OFF)
df_PD_OFFvsON_Act['w_ActONVsOFF'] = (df_PD_ON_Act['weight_parameter_map_magnitude']- df_PD_OFF_Act['weight_parameter_map_magnitude'])

df_PD_OFFvsON_Act['total_UPDRS_ONVsOFF'] = (df_PD_ON_Act['total_UPDRS']- df_PD_OFF_Act['total_UPDRS'])

df_PD_OFFvsON_Act['relVsIrHighRewardOption_ActONVsOFF'] = (df_PD_ON_Act['relevantVsIrrelevantHighRewardOption']- 
                                                           df_PD_OFF_Act['relevantVsIrrelevantHighRewardOption'])

df_PD_OFFvsON_Act['relHighRewardOption_ActONVsOFF'] = (df_PD_ON_Act['relevantHighRewardOption']- 
                                                       df_PD_OFF_Act['relevantHighRewardOption'])

# save in csv 
df_PD_OFFvsON_Act.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_PD_OFFvsON_Act.csv', index=False )

In [7]:
# PD OFF Act
df_PD_OFF_Act = df_all[(df_all['patient'] == 'PD') &(df_all['medication'] == 'OFF') &(
    df_all['condition'] == 'Act')].copy().reset_index(drop=True)

# PD OFF Stim
df_PD_OFF_Stim = df_all[(df_all['patient'] == 'PD') &(df_all['medication'] == 'OFF') &
    (df_all['condition'] == 'Clr')].copy().reset_index(drop=True)

# Ensure same subject order
df_PD_OFF_Act = df_PD_OFF_Act.sort_values('sub_ID').reset_index(drop=True)
df_PD_OFF_Stim = df_PD_OFF_Stim.sort_values('sub_ID').reset_index(drop=True)

# Base dataframe
df_PD_OFF_Stim_Act = df_PD_OFF_Act[['patient', 'medication', 'sub_ID', 'condition','x_number', 'age',
    'sex', 'disease_duration', 'time_symptomns','most_affected_left_right_side_baselineUPDRS',
    'most_affected_left_right_side_UPDRS','systolic_blood_pressure_baselineUPDRS',
    'total_BaselineUPDRS','total_UPDRS','NMSS', 'MoCA', 'BDI', 'LARS']].copy()

# Stim vs Act contrasts
df_PD_OFF_Stim_Act['w_ClrVsAct'] = (df_PD_OFF_Stim['weight_parameter_map_magnitude']
                                    - df_PD_OFF_Act['weight_parameter_map_magnitude'])

df_PD_OFF_Stim_Act['relVsIrHighRewardOption_ClrVsAct'] = (df_PD_OFF_Stim['relevantVsIrrelevantHighRewardOption']
                                                          - df_PD_OFF_Act['relevantVsIrrelevantHighRewardOption'])

df_PD_OFF_Stim_Act['relHighRewardOption_ClrVsAct'] = (df_PD_OFF_Stim['relevantHighRewardOption']
                                                      - df_PD_OFF_Act['relevantHighRewardOption'])
# save in csv 
df_PD_OFF_Stim_Act.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_PD_OFF_Stim_Act.csv', index=False )

In [9]:
# Save HC group seperately
df_HC = df_all[(df_all['patient'] == 'HC')].copy().reset_index(drop=True)
df_HC.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_HC.csv', index=False )

# Save PD group seperately
df_HC = df_all[(df_all['patient'] == 'PD')].copy().reset_index(drop=True)
df_HC.to_csv(f'{PROJECT_CLIN_EVAL_DIR}/clinical_eval_parameter_PD.csv', index=False )